In [ ]:
# -*- coding: utf-8 -*-
"""
SKYVIDYA Agnostic Content Generation Framework v2.0

Este notebook implementa a arquitetura de agentes de IA para gerar relatórios
e conteúdos de marketing de forma agnóstica ao caso de uso.

Novas funcionalidades:
- Integração com APIs de LLMs (Google Gemini, OpenAI, Anthropic).
- Gestão de workflow e aprovações através de uma Tabela de Controlo no Google Sheets.
"""

# --- Célula 1: Instalações e Configuração Inicial ---
# Descomente e execute esta célula uma vez para instalar as dependências.
# print("Instalando dependências...")
# !pip install pandas geopandas pyarrow matplotlib seaborn --quiet
# !pip install google-api-python-client google-auth-httplib2 google-auth-oauthlib --quiet
!pip install google-generativeai openai anthropic --quiet
# print("Instalação concluída.")

import json
import pandas as pd
import geopandas as gpd
import os
from IPython.display import display, HTML, Image

# Bibliotecas de IA e APIs
import google.generativeai as genai
import openai
import anthropic

from google.colab import drive
drive.mount('/content/drive/',force_remount=True)

# Bibliotecas para a Tabela de Controlo (Google Sheets)
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# --- Célula 2: Configuração de Chaves de API (Segurança) ---
# Use o gestor de segredos do Google Colab para armazenar as suas chaves de API de forma segura.
# No menu à esquerda, clique no ícone de chave (secrets) e adicione os seguintes segredos:
# GOOGLE_API_KEY  -> A sua chave da API do Google (para Gemini)
# OPENAI_API_KEY  -> A sua chave da API da OpenAI (para ChatGPT)
# ANTHROPIC_API_KEY -> A sua chave da API da Anthropic (para Claude)

from google.colab import userdata

try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')

    genai.configure(api_key=GOOGLE_API_KEY)
    openai.api_key = OPENAI_API_KEY

    print("Chaves de API carregadas com sucesso.")
except Exception as e:
    print("AVISO: Não foi possível carregar todas as chaves de API. O agente LLMService poderá não funcionar para todos os provedores.")
    print("Por favor, configure os segredos 'GOOGLE_API_KEY', 'OPENAI_API_KEY', e 'ANTHROPIC_API_KEY' no Google Colab.")


# --- Célula 3: Definição dos Agentes de IA (Classes) ---

class LLMService:
    """Serviço de abstração para interagir com diferentes modelos de LLM."""
    def __init__(self):
        self.anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) if 'ANTHROPIC_API_KEY' in globals() else None

    def generate(self, prompt, provider='gemini', model=None):
        print(f"INFO: [LLMService] Enviando prompt para o provedor: {provider.upper()}")
        try:
            if provider == 'gemini':
                model = genai.GenerativeModel(model or 'gemini-2.5-flash')
                response = model.generate_content(prompt)
                return response.text
            elif provider == 'openai':
                response = openai.chat.completions.create(
                    model=model or "gpt-4o",
                    messages=[{"role": "user", "content": prompt}]
                )
                return response.choices[0].message.content
            elif provider == 'anthropic':
                if not self.anthropic_client:
                    raise ConnectionError("Cliente da Anthropic não foi inicializado. Verifique a chave de API.")
                message = self.anthropic_client.messages.create(
                    model=model or "claude-3-sonnet-20240229",
                    max_tokens=4000,
                    messages=[{"role": "user", "content": prompt}]
                )
                return message.content[0].text
            else:
                raise ValueError(f"Provedor de LLM desconhecido: {provider}")
        except Exception as e:
            print(f"ERRO: [LLMService] Falha ao gerar conteúdo com {provider.upper()}. Erro: {e}")
            return f"ERRO NA GERAÇÃO DE CONTEÚDO COM {provider.upper()}."

class WorkflowManager:
    """
    Agente de Workflow: Gere o estado da produção de conteúdo
    usando uma Tabela de Controlo no Google Sheets.
    """
    def __init__(self, spreadsheet_id):
        self.spreadsheet_id = spreadsheet_id
        self.service = self._authenticate_and_build()
        self.sheet_title = ""

    def _authenticate_and_build(self):
        """Autentica o utilizador e constrói o serviço do Google Sheets."""
        try:
            auth.authenticate_user()
            return build('sheets', 'v4')
        except Exception as e:
            print(f"ERRO: [WorkflowManager] Falha na autenticação com o Google. {e}")
            return None

    def _get_sheet_id(self, sheet_title):
        """Obtém o ID de uma aba pelo seu título."""
        if not self.service: return None
        try:
            sheet_metadata = self.service.spreadsheets().get(spreadsheetId=self.spreadsheet_id).execute()
            sheets = sheet_metadata.get('sheets', '')
            for sheet in sheets:
                if sheet.get("properties", {}).get("title", "") == sheet_title:
                    return sheet.get("properties", {}).get("sheetId", "")
        except HttpError as e:
            print(f"ERRO: [WorkflowManager] Não foi possível obter metadados da planilha. {e}")
        return None

    def setup_control_sheet(self, sheet_title):
        """Verifica se a aba do caso de uso existe; se não, cria."""
        if not self.service: return False
        self.sheet_title = sheet_title
        print(f"INFO: [WorkflowManager] Verificando/Configurando a aba de controlo: '{sheet_title}'")
        if self._get_sheet_id(sheet_title) is None:
            print(f"INFO: [WorkflowManager] Aba '{sheet_title}' não encontrada. A criar...")
            try:
                body = {'requests': [{'addSheet': {'properties': {'title': sheet_title}}}]}
                self.service.spreadsheets().batchUpdate(spreadsheetId=self.spreadsheet_id, body=body).execute()

                # Adiciona o cabeçalho
                header = [["ID do Conteúdo", "Formato", "Estado", "Data de Criação", "Link para Revisão/Output"]]
                self.service.spreadsheets().values().update(
                    spreadsheetId=self.spreadsheet_id,
                    range=f"'{sheet_title}'!A1",
                    valueInputOption='USER_ENTERED',
                    body={'values': header}
                ).execute()
                print(f"INFO: [WorkflowManager] Aba '{sheet_title}' criada com sucesso.")
            except HttpError as e:
                print(f"ERRO: [WorkflowManager] Falha ao criar a aba. {e}")
                return False
        else:
            print(f"INFO: [WorkflowManager] Aba '{sheet_title}' já existe.")
        return True

    def add_content_jobs(self, formats_to_generate, case_name):
        """Adiciona novas tarefas à tabela de controlo."""
        if not self.service or not self.sheet_title: return
        print(f"INFO: [WorkflowManager] Adicionando tarefas de conteúdo à Tabela de Controlo...")

        timestamp = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
        rows = []
        for fmt in formats_to_generate:
            content_id = f"{case_name.replace(' ', '_')}_{fmt}"
            rows.append([content_id, fmt, "Em desenvolvimento", timestamp, ""])

        body = {'values': rows}
        try:
            self.service.spreadsheets().values().append(
                spreadsheetId=self.spreadsheet_id,
                range=f"'{self.sheet_title}'!A:E",
                valueInputOption='USER_ENTERED',
                insertDataOption='INSERT_ROWS',
                body=body
            ).execute()
            print(f"INFO: [WorkflowManager] {len(rows)} tarefas adicionadas com sucesso.")
        except HttpError as e:
            print(f"ERRO: [WorkflowManager] Falha ao adicionar tarefas. {e}")

    def update_job_status(self, content_id, new_status, output_link=""):
        """Atualiza o estado de uma tarefa específica na tabela."""
        if not self.service or not self.sheet_title: return
        print(f"INFO: [WorkflowManager] Atualizando estado de '{content_id}' para '{new_status}'...")
        try:
            # Primeiro, encontra a linha correspondente ao content_id
            result = self.service.spreadsheets().values().get(
                spreadsheetId=self.spreadsheet_id,
                range=f"'{self.sheet_title}'!A:A"
            ).execute()
            values = result.get('values', [])
            row_index = -1
            for i, row in enumerate(values):
                if row and row[0] == content_id:
                    row_index = i + 1 # +1 porque a API é baseada em 1
                    break

            if row_index == -1:
                print(f"AVISO: [WorkflowManager] ID '{content_id}' não encontrado para atualização.")
                return

            # Atualiza a coluna de estado (C) e de link (E)
            update_body = {'values': [[new_status, pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'), output_link]]} # Inclui data de atualização
            self.service.spreadsheets().values().update(
                spreadsheetId=self.spreadsheet_id,
                range=f"'{self.sheet_title}'!C{row_index}:E{row_index}",
                valueInputOption='USER_ENTERED',
                body=update_body
            ).execute()
            print(f"INFO: [WorkflowManager] Estado de '{content_id}' atualizado com sucesso.")
        except HttpError as e:
            print(f"ERRO: [WorkflowManager] Falha ao atualizar estado. {e}")


class DataSynthesizer:
    # ... (código inalterado) ...
    def __init__(self, config): self.config = config; self.gdf = None

    def load_data(self):
        path = self.config['data_file_path']
        print(f"INFO: [DataSynthesizer] Carregando dados de: {path}");
        try:
            self.gdf = gpd.read_parquet(path)
            if self.config.get('region'): self.gdf = self.gdf[self.gdf['SIGLA_UF'] == self.config['region']]
            print("INFO: [DataSynthesizer] Dados carregados e filtrados com sucesso.")
        except Exception as e: print(f"ERRO: [DataSynthesizer] Falha ao carregar dados. {e}"); return False
        return True

    def extract_kpis(self):
        if self.gdf is None or self.gdf.empty: print("ERRO: [DataSynthesizer] Dados não disponíveis para extração de KPIs."); return None
        print("INFO: [DataSynthesizer] Extraindo KPIs...")
        metrics = self.config['key_metrics']
        risk_col = metrics['risk_category_col']; high_risk_count = self.gdf[self.gdf[risk_col].isin(['Alto', 'Muito Alto'])].shape[0]
        trend_col = metrics['trend_col']; growing_trend_count = self.gdf[self.gdf[trend_col] == 'Crescente'].shape[0]
        threat_col = metrics['main_threat_col']; main_threat = self.gdf[threat_col].value_counts().idxmax()
        kpis = {"total_municipalities": self.gdf.shape[0], "percent_high_risk": round((high_risk_count / self.gdf.shape[0]) * 100) if self.gdf.shape[0] > 0 else 0, "percent_growing_trend": round((growing_trend_count / self.gdf.shape[0]) * 100) if self.gdf.shape[0] > 0 else 0, "dominant_threat": main_threat, "top_risk_municipalities": self.gdf.nlargest(3, metrics['risk_score_col'])['NM_MUN_SEM_ACENTO'].tolist()}
        print(f"INFO: [DataSynthesizer] KPIs extraídos: {kpis}"); return kpis

class MultimediaArtDirector:
    # ... (código inalterado) ...
    def generate_visual_prompts(self, kpis, config):
        print("INFO: [ArtDirector] Gerando prompts para visuais...")
        prompts = {"cover_image_prompt": f"Capa para relatório de risco, tema '{config['ecosystem']}: {config['case_name']}' em '{config['region']}'. Abstração de mapa topográfico com dados. Vertical 9:16.", "blog_header_prompt": f"Cabeçalho de blog sobre '{config['case_name']}'. Foto aérea de '{config['region']}', metade próspera, metade protegida por escudo digital contra tempestade. Realista. 16:9.", "linkedin_infographic_prompt": f"Infográfico 3D elegante para o LinkedIn. Mapa 3D de '{config['region']}' com risco destacado. Texto: '{kpis['percent_high_risk']}% em Risco Alto'. Tema '{config['case_name']}'. 1:1.", "instagram_map_prompt": f"Mapa de alerta para Instagram de '{config['region']}'. Heatmap de risco vermelho/laranja. Título 'MAPA DO RISCO: {config['case_name'].upper()}'. Foco em '{kpis['dominant_threat']}'. 1:1."}
        print("INFO: [ArtDirector] Prompts gerados."); return prompts

    def generate_visual_assets(self, prompts):
        print("INFO: [ArtDirector] SIMULANDO geração de imagens...");
        assets = {"cover_image_url": "https://storage.googleapis.com/gemini-generative-ai-docs/skyvidya/skyvidya_cover.png", "blog_header_url": "https://storage.googleapis.com/gemini-generative-ai-docs/skyvidya/skyvidya_blog.png", "linkedin_infographic_url": "https://storage.googleapis.com/gemini-generative-ai-docs/skyvidya/skyvidya_linkedin.png", "instagram_map_url": "https://storage.googleapis.com/gemini-generative-ai-docs/skyvidya/skyvidya_instagram.png"}
        print("INFO: [ArtDirector] URLs de ativos simulados retornados."); return assets

class LLMCopywriter:
    """
    Agente 4: Gerador de Conteúdo Textual (com Integração de API)
    Gera o texto para os diferentes formatos usando um LLM.
    """
    def __init__(self):
        self.llm_service = LLMService()

    def _create_prompt(self, format_type, kpis, config):
        """Cria um prompt detalhado para o LLM com base no formato."""
        # ... (código dos prompts do _generate_..._text anterior, agora formatado como um prompt)
        if format_type == "full_report":
             return f"""
                Você é um analista de risco da SKYVIDYA. Escreva a introdução e o sumário executivo de um relatório completo.
                Título: SKYVIDYA INSIGHTS - {config['ecosystem'].upper()} / {config['region'].upper()} / {config.get('year', 2025)} - {config['case_name']}

                KPIs Chave:
                - Total de Municípios Analisados: {kpis['total_municipalities']}
                - Percentagem em Risco Alto/Muito Alto: {kpis['percent_high_risk']}%
                - Percentagem com Tendência Crescente: {kpis['percent_growing_trend']}%
                - Ameaça Dominante: {kpis['dominant_threat']}
                - Top 3 Municípios em Risco: {', '.join(kpis['top_risk_municipalities'])}

                Estrutura a seguir: PREFÁCIO, SUMÁRIO EXECUTIVO (com Visão Geral, Principais Descobertas e Implicações Estratégicas).
                Use um tom formal e analítico.
             """
        # ... (prompts similares para os outros formatos) ...
        elif format_type == "blog_post":
            return f"""
                Você é um redator de conteúdo para o blog da SKYVIDYA. Escreva um post de blog envolvente.
                Tema: {config['case_name']} no/a {config['region']}.
                Título Sugerido: Mapa do Risco: Descobrimos as Áreas Mais Vulneráveis a Desastres no Rio Grande do Sul.

                Pontos a destacar:
                - Ameaça principal: {kpis['dominant_threat']}.
                - Insight chocante: {kpis['percent_high_risk']}% dos municípios estão em risco alto ou muito alto.
                - Urgência: {kpis['percent_growing_trend']}% dos municípios têm tendência crescente de eventos.

                Use um tom acessível e termine com uma chamada para a ação para ler o relatório completo.
            """
        return "Prompt Padrão: Resuma os dados." # Fallback

    def generate_all_content(self, kpis, config, assets):
        print("INFO: [Copywriter] Gerando conteúdo textual via API LLM...")
        content = {}
        requested_outputs = config.get('outputs', [])
        llm_provider = config.get('llm_provider', 'gemini') # Default para gemini

        for output_type in requested_outputs:
            prompt = self._create_prompt(output_type, kpis, config)
            generated_text = self.llm_service.generate(prompt, provider=llm_provider)
            content[output_type] = generated_text # HTML ou Markdown dependendo do LLM

        print("INFO: [Copywriter] Conteúdo textual gerado.")
        return content


class ReportAssembler:
    # ... (código inalterado) ...
    def assemble_for_review(self, content, assets, config):
        print("\n" + "="*80); print("PARA APROVAÇÃO HUMANA: CONTEÚDO GERADO PELO FRAMEWORK AGNÓSTICO"); print("="*80 + "\n")
        for format_name, text in content.items():
            display(HTML(f"<h3>FORMATO: {format_name.replace('_',' ').upper()}</h3>"))
            if "report" in format_name: display(HTML(f'<img src="{assets["cover_image_url"]}" style="width: 300px; border: 1px solid #ccc;" />')); display(HTML(text))
            elif "blog" in format_name: display(HTML(f'<img src="{assets["blog_header_url"]}" style="width: 500px; border: 1px solid #ccc;" />')); display(HTML(text))
            elif "linkedin" in format_name: display(HTML(f'<img src="{assets["linkedin_infographic_url"]}" style="width: 300px; border: 1px solid #ccc;" />')); display(HTML(f"<pre style='white-space: pre-wrap; word-wrap: break-word; background-color: #f0f0f0; padding: 10px; border-radius: 5px;'>{text}</pre>"))
            elif "instagram" in format_name: display(HTML(f'<img src="{assets["instagram_map_url"]}" style="width: 300px; border: 1px solid #ccc;" />')); display(HTML(f"<pre style='white-space: pre-wrap; word-wrap: break-word; background-color: #f0f0f0; padding: 10px; border-radius: 5px;'>{text}</pre>"))
        print("\n" + "="*80); print("APROVAÇÃO SOLICITADA: Por favor, reveja o conteúdo gerado acima."); print("="*80 + "\n")


# --- Célula 4: Configuração do Caso de Uso e Orquestração ---

# Este dicionário de configuração é o ÚNICO item que precisa ser alterado
# para executar o framework para um novo caso de uso.
config_risco_desastres_rs = {
  "case_name": "Risco de Desastres Geohidrológicos",
  "ecosystem": "Climático",
  "region": "RS",
  "spreadsheet_id": "1hriMWNFvVtKn2i7OUaaQAV__ZNRCjqkcAea_2FzLwWY", # <--- AÇÃO NECESSÁRIA
  "data_file_path": "/content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/03_gold/climate_risk_final_analysis.geoparquet",
  "llm_provider": "gemini", # Escolha entre 'gemini', 'openai', 'anthropic'
  "key_metrics": {
    "risk_category_col": "Risco_Ampliado_MCDA_Cat",
    "risk_score_col": "Risco_Ampliado_MCDA_Score",
    "trend_col": "Tendencia_Eventos_Climaticos_Extremos",
    "main_threat_col": "principal_ameaca",
    "lisa_cluster_col": "LAST10_YEARS_COUNT_lisa_labels"
  },
  "outputs": ["full_report", "report_summary", "blog_post", "linkedin_post"]
}


def run_framework(config):
    """Função principal que orquestra a execução dos agentes."""

    print(f"--- INICIANDO FRAMEWORK PARA O CASO: {config['case_name']} EM {config['region']} ---")

    # 1. Gestor de Workflow
    if not config.get("spreadsheet_id") or "COLE_AQUI" in config.get("spreadsheet_id"):
        print("ERRO: ID da Planilha Google não configurado. Por favor, atualize o 'spreadsheet_id' na configuração.")
        return
    workflow_manager = WorkflowManager(config['spreadsheet_id'])
    sheet_title = f"{config['region']}_{config['ecosystem']}_{config['case_name']}".replace(' ', '_')
    if not workflow_manager.setup_control_sheet(sheet_title):
        return # Para se a configuração da planilha falhar
    workflow_manager.add_content_jobs(config['outputs'], config['case_name'])

    # 2. Sintetizador de Dados
    data_synthesizer = DataSynthesizer(config)
    if not data_synthesizer.load_data(): return
    kpis = data_synthesizer.extract_kpis()
    if not kpis: return

    # 3. Diretor de Arte
    art_director = MultimediaArtDirector()
    prompts = art_director.generate_visual_prompts(kpis, config)
    assets = art_director.generate_visual_assets(prompts)

    # 4. Gerador de Conteúdo
    copywriter = LLMCopywriter()
    content = copywriter.generate_all_content(kpis, config, assets)

    # 5. Atualiza Workflow e Monta para Revisão
    assembler = ReportAssembler()
    assembler.assemble_for_review(content, assets, config)
    for fmt in config['outputs']:
        content_id = f"{config['case_name'].replace(' ', '_')}_{fmt}"
        # Em um sistema real, o link seria para o rascunho salvo (ex: Google Docs)
        workflow_manager.update_job_status(content_id, "Aguardando Aprovação", "Revisar output no notebook de execução.")

# --- Célula 5: Execução ---
run_framework(config_risco_desastres_rs) # Descomente para executar

print("Framework de Agentes definido. Para executar, preencha o 'spreadsheet_id' na configuração e descomente a linha 'run_framework(config_risco_desastres_rs)'.")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Mounted at /content/drive/
Chaves de API carregadas com sucesso.
--- INICIANDO FRAMEWORK PARA O CASO: Risco de Desastres Geohidrológicos EM RS ---
INFO: [WorkflowManager] Verificando/Configurando a aba de controlo: 'RS_Climático_Risco_de_Desastres_Geohidrológicos'


INFO: [WorkflowManager] Aba 'RS_Climático_Risco_de_Desastres_Geohidrológicos' já existe.
INFO: [WorkflowManager] Adicionando tarefas de conteúdo à Tabela de Controlo...
INFO: [WorkflowManager] 4 tarefas adicionadas com sucesso.
INFO: [DataSynthesizer] Carregando dados de: /content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/03_gold/climate_risk_final_analysis.geoparquet
INFO: [DataSynthesizer] Dados carregados e filtrados com sucesso.
INFO: [DataSynthesizer] Extraindo KPIs...
INFO: [DataSynthesizer] KPIs extraídos: {'total_municipalities': 499, 'percent_high_risk': 40, 'percent_growing_trend': 62, 'dominant_threat': 'Chuvas Intensas', 'top_risk_municipalities': ['PONTE PRETA', 'GENTIL', 'BARRA DO RIO AZUL']}
INFO: [ArtDirector] Gerando prompts para visuais...
INFO: [ArtDirector] Prompts gerados.
INFO: [ArtDirector] SIMULANDO geração de imagens...
INFO: [ArtDirector] URLs de ativos simulados retornados.
INFO: [Copywriter] Gerando conteúdo textual via API LLM...
INFO: [


APROVAÇÃO SOLICITADA: Por favor, reveja o conteúdo gerado acima.

INFO: [WorkflowManager] Atualizando estado de 'Risco_de_Desastres_Geohidrológicos_full_report' para 'Aguardando Aprovação'...
INFO: [WorkflowManager] Estado de 'Risco_de_Desastres_Geohidrológicos_full_report' atualizado com sucesso.
INFO: [WorkflowManager] Atualizando estado de 'Risco_de_Desastres_Geohidrológicos_report_summary' para 'Aguardando Aprovação'...
INFO: [WorkflowManager] Estado de 'Risco_de_Desastres_Geohidrológicos_report_summary' atualizado com sucesso.
INFO: [WorkflowManager] Atualizando estado de 'Risco_de_Desastres_Geohidrológicos_blog_post' para 'Aguardando Aprovação'...
INFO: [WorkflowManager] Estado de 'Risco_de_Desastres_Geohidrológicos_blog_post' atualizado com sucesso.
INFO: [WorkflowManager] Atualizando estado de 'Risco_de_Desastres_Geohidrológicos_linkedin_post' para 'Aguardando Aprovação'...
INFO: [WorkflowManager] Estado de 'Risco_de_Desastres_Geohidrológicos_linkedin_post' atualizado com suce

In [ ]:
# -*- coding: utf-8 -*-
"""
SKYVIDYA Agnostic Content Generation Framework v3.0

Esta versão completa integra as melhorias da v2 (APIs de LLM, Google Sheets)
com as implementações de código que estavam faltando, restauradas da v1.
Cada agente é representado por uma classe Python, e o fluxo é controlado
por um dicionário de configuração flexível e escalável.

Novas funcionalidades (v2) mantidas:
- Integração com APIs de LLMs (Google Gemini, OpenAI, Anthropic).
- Gestão de workflow e aprovações através de uma Tabela de Controlo no Google Sheets.

Funcionalidades da v1 restauradas:
- Implementações completas de todas as classes de agentes.
- Lógica detalhada de extração de KPIs e geração de prompts para visuais.
- Templates de conteúdo da v1 convertidos em prompts para LLMs.
"""

# --- Célula 1: Instalações e Configuração Inicial ---
# Descomente e execute esta célula uma vez para instalar as dependências.
# print("Instalando dependências...")
# !pip install pandas geopandas pyarrow matplotlib seaborn --quiet
# !pip install google-api-python-client google-auth-httplib2 google-auth-oauthlib --quiet
!pip install google-generativeai openai anthropic --quiet
# print("Instalação concluída.")

import json
import pandas as pd
import geopandas as gpd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML, Image

# Bibliotecas de IA e APIs
import google.generativeai as genai
import openai
import anthropic

from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

# Bibliotecas para a Tabela de Controlo (Google Sheets)
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# --- Célula 2: Configuração de Chaves de API (Segurança) ---
# Use o gestor de segredos do Google Colab para armazenar as suas chaves de API de forma segura.
# No menu à esquerda, clique no ícone de chave (secrets) e adicione os seguintes segredos:
# GOOGLE_API_KEY  -> A sua chave da API do Google (para Gemini)
# OPENAI_API_KEY  -> A sua chave da API da OpenAI (para ChatGPT)
# ANTHROPIC_API_KEY -> A sua chave da API da Anthropic (para Claude)

from google.colab import userdata

try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')

    genai.configure(api_key=GOOGLE_API_KEY)
    openai.api_key = OPENAI_API_KEY

    print("Chaves de API carregadas com sucesso.")
except Exception as e:
    print("AVISO: Não foi possível carregar todas as chaves de API. O agente LLMService poderá não funcionar para todos os provedores.")
    print("Por favor, configure os segredos 'GOOGLE_API_KEY', 'OPENAI_API_KEY', e 'ANTHROPIC_API_KEY' no Google Colab.")


# --- Célula 3: Definição dos Agentes de IA (Classes) ---

class LLMService:
    """Serviço de abstração para interagir com diferentes modelos de LLM."""
    def __init__(self):
        self.anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) if 'ANTHROPIC_API_KEY' in globals() else None

    def generate(self, prompt, provider='gemini', model=None):
        print(f"INFO: [LLMService] Enviando prompt para o provedor: {provider.upper()}")
        try:
            if provider == 'gemini':
                model = genai.GenerativeModel(model or 'gemini-2.5-flash')
                response = model.generate_content(prompt)
                return response.text
            elif provider == 'openai':
                response = openai.chat.completions.create(
                    model=model or "gpt-4o",
                    messages=[{"role": "user", "content": prompt}]
                )
                return response.choices[0].message.content
            elif provider == 'anthropic':
                if not self.anthropic_client:
                    raise ConnectionError("Cliente da Anthropic não foi inicializado. Verifique a chave de API.")
                message = self.anthropic_client.messages.create(
                    model=model or "claude-3-sonnet-20240229",
                    max_tokens=4000,
                    messages=[{"role": "user", "content": prompt}]
                )
                return message.content[0].text
            else:
                raise ValueError(f"Provedor de LLM desconhecido: {provider}")
        except Exception as e:
            print(f"ERRO: [LLMService] Falha ao gerar conteúdo com {provider.upper()}. Erro: {e}")
            return f"ERRO NA GERAÇÃO DE CONTEÚDO COM {provider.upper()}."

class WorkflowManager:
    """
    Agente de Workflow: Gere o estado da produção de conteúdo
    usando uma Tabela de Controlo no Google Sheets.
    """
    def __init__(self, spreadsheet_id):
        self.spreadsheet_id = spreadsheet_id
        self.service = self._authenticate_and_build()
        self.sheet_title = ""

    def _authenticate_and_build(self):
        """Autentica o utilizador e constrói o serviço do Google Sheets."""
        try:
            auth.authenticate_user()
            return build('sheets', 'v4')
        except Exception as e:
            print(f"ERRO: [WorkflowManager] Falha na autenticação com o Google. {e}")
            return None

    def _get_sheet_id(self, sheet_title):
        """Obtém o ID de uma aba pelo seu título."""
        if not self.service: return None
        try:
            sheet_metadata = self.service.spreadsheets().get(spreadsheetId=self.spreadsheet_id).execute()
            sheets = sheet_metadata.get('sheets', '')
            for sheet in sheets:
                if sheet.get("properties", {}).get("title", "") == sheet_title:
                    return sheet.get("properties", {}).get("sheetId", "")
        except HttpError as e:
            print(f"ERRO: [WorkflowManager] Não foi possível obter metadados da planilha. {e}")
        return None

    def setup_control_sheet(self, sheet_title):
        """Verifica se a aba do caso de uso existe; se não, cria."""
        if not self.service: return False
        self.sheet_title = sheet_title
        print(f"INFO: [WorkflowManager] Verificando/Configurando a aba de controlo: '{sheet_title}'")
        if self._get_sheet_id(sheet_title) is None:
            print(f"INFO: [WorkflowManager] Aba '{sheet_title}' não encontrada. A criar...")
            try:
                body = {'requests': [{'addSheet': {'properties': {'title': sheet_title}}}]}
                self.service.spreadsheets().batchUpdate(spreadsheetId=self.spreadsheet_id, body=body).execute()

                header = [["ID do Conteúdo", "Formato", "Estado", "Data de Atualização", "Link para Revisão/Output"]]
                self.service.spreadsheets().values().update(
                    spreadsheetId=self.spreadsheet_id,
                    range=f"'{sheet_title}'!A1",
                    valueInputOption='USER_ENTERED',
                    body={'values': header}
                ).execute()
                print(f"INFO: [WorkflowManager] Aba '{sheet_title}' criada com sucesso.")
            except HttpError as e:
                print(f"ERRO: [WorkflowManager] Falha ao criar a aba. {e}")
                return False
        else:
            print(f"INFO: [WorkflowManager] Aba '{sheet_title}' já existe.")
        return True

    def add_content_jobs(self, formats_to_generate, case_name):
        """Adiciona novas tarefas à tabela de controlo."""
        if not self.service or not self.sheet_title: return
        print(f"INFO: [WorkflowManager] Adicionando tarefas de conteúdo à Tabela de Controlo...")

        timestamp = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
        rows = []
        for fmt in formats_to_generate:
            content_id = f"{case_name.replace(' ', '_')}_{fmt}"
            # Adiciona com estado inicial "Pendente"
            rows.append([content_id, fmt, "Pendente", timestamp, ""])

        body = {'values': rows}
        try:
            self.service.spreadsheets().values().append(
                spreadsheetId=self.spreadsheet_id,
                range=f"'{self.sheet_title}'!A:E",
                valueInputOption='USER_ENTERED',
                insertDataOption='INSERT_ROWS',
                body=body
            ).execute()
            print(f"INFO: [WorkflowManager] {len(rows)} tarefas adicionadas com sucesso.")
        except HttpError as e:
            print(f"ERRO: [WorkflowManager] Falha ao adicionar tarefas. {e}")

    def update_job_status(self, content_id, new_status, output_link=""):
        """Atualiza o estado de uma tarefa específica na tabela."""
        if not self.service or not self.sheet_title: return
        print(f"INFO: [WorkflowManager] Atualizando estado de '{content_id}' para '{new_status}'...")
        try:
            result = self.service.spreadsheets().values().get(
                spreadsheetId=self.spreadsheet_id,
                range=f"'{self.sheet_title}'!A:A"
            ).execute()
            values = result.get('values', [])
            row_index = -1
            for i, row in enumerate(values):
                if row and row[0] == content_id:
                    row_index = i + 1
                    break

            if row_index == -1:
                print(f"AVISO: [WorkflowManager] ID '{content_id}' não encontrado para atualização.")
                return

            update_body = {'values': [[new_status, pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'), output_link]]}
            self.service.spreadsheets().values().update(
                spreadsheetId=self.spreadsheet_id,
                range=f"'{self.sheet_title}'!C{row_index}:E{row_index}",
                valueInputOption='USER_ENTERED',
                body=update_body
            ).execute()
            print(f"INFO: [WorkflowManager] Estado de '{content_id}' atualizado com sucesso.")
        except HttpError as e:
            print(f"ERRO: [WorkflowManager] Falha ao atualizar estado. {e}")

class DataSynthesizer:
    """
    Agente 2: Sintetizador de Dados
    Carrega os dados e extrai os KPIs numéricos e textuais necessários.
    """
    def __init__(self, config):
        self.config = config
        self.gdf = None

    def load_data(self):
        """Carrega o arquivo GeoParquet especificado na configuração."""
        path = self.config['data_file_path']
        print(f"INFO: [DataSynthesizer] Carregando dados de: {path}")
        try:
            self.gdf = gpd.read_parquet(path)
            if self.config.get('region'):
                self.gdf = self.gdf[self.gdf['SIGLA_UF'] == self.config['region']]
            print("INFO: [DataSynthesizer] Dados carregados e filtrados com sucesso.")
        except Exception as e:
            print(f"ERRO: [DataSynthesizer] Falha ao carregar dados. {e}")
            return False
        return True

    def extract_kpis(self):
        """Extrai os KPIs definidos na configuração."""
        if self.gdf is None or self.gdf.empty:
            print("ERRO: [DataSynthesizer] Dados não disponíveis para extração de KPIs.")
            return None

        print("INFO: [DataSynthesizer] Extraindo KPIs...")
        metrics = self.config['key_metrics']

        risk_col = metrics['risk_category_col']
        high_risk_count = self.gdf[self.gdf[risk_col].isin(['Alto', 'Muito Alto'])].shape[0]

        trend_col = metrics['trend_col']
        growing_trend_count = self.gdf[self.gdf[trend_col] == 'Crescente'].shape[0]

        threat_col = metrics['main_threat_col']
        main_threat = self.gdf[threat_col].value_counts().idxmax()

        kpis = {
            "total_municipalities": self.gdf.shape[0],
            "percent_high_risk": round((high_risk_count / self.gdf.shape[0]) * 100) if self.gdf.shape[0] > 0 else 0,
            "percent_growing_trend": round((growing_trend_count / self.gdf.shape[0]) * 100) if self.gdf.shape[0] > 0 else 0,
            "dominant_threat": main_threat,
            "top_risk_municipalities": self.gdf.nlargest(3, metrics['risk_score_col'])['NM_MUN_SEM_ACENTO'].tolist()
        }
        print(f"INFO: [DataSynthesizer] KPIs extraídos: {kpis}")
        return kpis

class MultimediaArtDirector:
    """
    Agente 3: Diretor de Arte Multimédia (Simulado)
    Gera prompts para os ativos visuais e retorna URLs de placeholder.
    """
    def generate_visual_prompts(self, kpis, config):
        """Gera prompts com base nos KPIs e na configuração."""
        print("INFO: [ArtDirector] Gerando prompts para visuais...")
        prompts = {
            "cover_image_prompt": f"Uma imagem de capa para um relatório de inteligência de risco, estilo corporativo e sofisticado. O tema é '{config['ecosystem']}: {config['case_name']}' na região de '{config['region']}'. A imagem deve ser uma abstração artística de um mapa topográfico com sobreposições de dados. Formato vertical 9:16.",
            "blog_header_prompt": f"Imagem de cabeçalho para post de blog sobre '{config['case_name']}'. Mostrar uma fotografia aérea de '{config['region']}' dividida ao meio: um lado próspero, o outro protegido por um escudo digital que reflete dados sobre '{kpis['dominant_threat']}'. Estilo fotorrealista. Formato 16:9.",
            "linkedin_infographic_prompt": f"Infográfico 3D elegante para o LinkedIn. Mapa 3D do/a '{config['region']}' com áreas de risco destacadas. Texto em destaque: '{kpis['percent_high_risk']}% dos municípios em Risco Alto'. Foco no tema '{config['case_name']}'. Formato quadrado 1:1.",
            "instagram_map_prompt": f"Mapa de alerta de emergência do/a '{config['region']}' para o Instagram. Áreas de risco 'Muito Alto' e 'Alto' brilhando em vermelho e laranja. Título: 'MAPA DO RISCO: {config['case_name'].upper()}'. Foco em '{kpis['dominant_threat']}'. Formato 1:1."
        }
        print("INFO: [ArtDirector] Prompts gerados.")
        return prompts

    def generate_visual_assets(self, prompts):
        """Simula a geração de imagens e retorna URLs de placeholder."""
        print("INFO: [ArtDirector] SIMULANDO geração de imagens a partir dos prompts...")
        # Em um sistema real, aqui haveria chamadas a APIs como DALL-E ou Midjourney
        assets = {
            "cover_image_url": "https://storage.googleapis.com/gemini-generative-ai-docs/skyvidya/skyvidya_cover.png",
            "blog_header_url": "https://storage.googleapis.com/gemini-generative-ai-docs/skyvidya/skyvidya_blog.png",
            "linkedin_infographic_url": "https://storage.googleapis.com/gemini-generative-ai-docs/skyvidya/skyvidya_linkedin.png",
            "instagram_map_url": "https://storage.googleapis.com/gemini-generative-ai-docs/skyvidya/skyvidya_instagram.png"
        }
        print("INFO: [ArtDirector] URLs de ativos visuais simulados retornados.")
        return assets

class LLMCopywriter:
    """
    Agente 4: Gerador de Conteúdo Textual (com Integração de API)
    Gera o texto para os diferentes formatos usando um LLM.
    """
    def __init__(self):
        self.llm_service = LLMService()

    def _create_prompt(self, format_type, kpis, config):
        """Cria um prompt detalhado para o LLM com base no formato."""
        ecosystem = config['ecosystem']
        region = config['region']
        case_name = config['case_name']
        year = config.get('year', 2025)

        # Dados de KPI para o prompt
        kpi_data = f"""
        - Total de Municípios Analisados: {kpis['total_municipalities']}
        - Percentagem em Risco Alto/Muito Alto: {kpis['percent_high_risk']}%
        - Percentagem com Tendência Crescente de Eventos: {kpis['percent_growing_trend']}%
        - Ameaça Dominante na região: {kpis['dominant_threat']}
        - Top 3 Municípios em Risco (maior score): {', '.join(kpis['top_risk_municipalities'])}
        """

        if format_type == "full_report":
            return f"""
            Você é um analista sênior de risco da SKYVIDYA, especializado em relatórios de inteligência.
            Sua tarefa é escrever as seções iniciais de um relatório técnico detalhado em HTML.

            Use o seguinte Título: "SKYVIDYA INSIGHTS™ - Relatório Completo"
            Subtítulo: "{ecosystem.upper()} / {region.upper()} / {year} - {case_name}"

            Com base nos KPIs abaixo, gere o conteúdo para as seguintes seções:
            1.  **PREFÁCIO:** Um parágrafo curto sobre a importância da análise de riscos multidimensional com o SKYVIDYA FRAMEWORK™.
            2.  **SUMÁRIO EXECUTIVO:** Dividido em:
                -   **Visão Geral:** Um parágrafo sobre o cenário complexo e a necessidade de intervenção.
                -   **Principais Descobertas:** Uma lista de 4 pontos (bullet points) destacando os KPIs mais importantes.
                -   **Implicações Estratégicas:** Um placeholder como "[Análise detalhada das implicações estratégicas...]".

            KPIs para usar:
            {kpi_data}

            O tom deve ser formal, analítico e corporativo. A saída deve ser em HTML (use tags como <h1>, <h2>, <h4>, <p>, <ul>, <li>, <b>).
            """
        elif format_type == "report_summary":
            return f"""
            Você é um consultor de comunicação da SKYVIDYA.
            Sua tarefa é criar um sumário executivo (white paper resumido) em HTML.

            Use o seguinte Título: "SKYVIDYA INSIGHTS™ - Sumário Executivo"
            Subtítulo: "{ecosystem.upper()} / {region.upper()} / {year} - {case_name}"

            Destaque os dois KPIs mais impactantes em caixas separadas, com a porcentagem em tamanho grande.
            - KPI 1: {kpis['percent_high_risk']}% dos municípios em risco ALTO ou MUITO ALTO.
            - KPI 2: {kpis['percent_growing_trend']}% dos municípios com tendência CRESCENTE de eventos.

            Após os destaques, mencione a ameaça dominante ({kpis['dominant_threat']}) e as áreas de concentração de risco.
            Inclua uma seção sobre "IMPLICAÇÕES ESTRATÉGICAS" resumindo as recomendações.
            Termine com uma chamada para ação para aceder ao relatório completo.

            KPIs para referência:
            {kpi_data}

            O tom deve ser direto e focado em insights para tomada de decisão. A saída deve ser em HTML.
            """
        elif format_type == "blog_post":
            return f"""
            Você é um redator de conteúdo para o blog da SKYVIDYA.
            Sua tarefa é escrever um post de blog envolvente e informativo.

            Título Sugerido: "Mapa do Risco: Descobrimos as Áreas Mais Vulneráveis a {case_name} no/a {region}"

            O post deve começar com uma pergunta retórica para engajar o leitor (ex: "Você já se perguntou por que...?").
            Revele os resultados mais surpreendentes da análise:
            - {kpis['percent_high_risk']}% dos municípios em risco alto ou muito alto.
            - {kpis['percent_growing_trend']}% com tendência de piora.
            - Mencione a principal ameaça: {kpis['dominant_threat']}.

            O objetivo é traduzir dados complexos em uma narrativa acessível.
            Termine explicando como a inteligência de dados pode ajudar a construir um futuro mais resiliente.
            O tom deve ser acessível, jornalístico e um pouco alarmante, mas esperançoso. A saída pode ser em HTML simples.
            """
        elif format_type == "linkedin_post":
            return f"""
            Você é um especialista em marketing B2B e redes sociais da SKYVIDYA.
            Sua tarefa é escrever um post para o LinkedIn direcionado a líderes, gestores de risco e decisores políticos.

            O post deve ser conciso e impactante. Comece com o insight principal:
            - "{kpis['percent_high_risk']}% dos municípios no/a {region} já estão em risco 'Alto' ou 'Muito Alto' para {case_name}."

            Adicione o fator de urgência:
            - "Mais preocupante ainda, {kpis['percent_growing_trend']}% apresentam uma tendência crescente na frequência de eventos."

            Conclua com uma provocação estratégica sobre a necessidade de planejamento e resiliência.
            Termine com as seguintes hashtags: #InteligênciaDeRisco #ESG #{ecosystem} #{region.replace(' ','')} #Sustentabilidade #SKYVIDYA

            O tom deve ser profissional, direto e estratégico. A saída deve ser texto puro, com quebras de linha.
            """
        elif format_type == "instagram_post":
            return f"""
            Você é o gestor de mídias sociais da SKYVIDYA.
            Sua tarefa é escrever uma legenda para um post no Instagram, que acompanhará um mapa visual.

            A legenda deve ser curta, direta e fácil de ler. Comece com um cabeçalho em maiúsculas:
            - "MAPA DO RISCO: {region.upper()}!"

            Destaque os principais dados:
            - {kpis['percent_high_risk']}% dos municípios em risco ALTO ou MUITO ALTO.
            - Principal ameaça: {kpis['dominant_threat']}.
            - Mencione 2 dos municípios mais vulneráveis: {kpis['top_risk_municipalities'][0]} e {kpis['top_risk_municipalities'][1]}.

            Termine com uma pergunta para engajar a audiência (ex: "A sua cidade está preparada?").
            Inclua as seguintes hashtags: #RiscoClimatico #{region.replace(' ','')} #{kpis['dominant_threat'].replace(' ','')} #DefesaCivil #Sustentabilidade #SKYVIDYA

            O tom deve ser informativo, de alerta e voltado para o público geral. A saída deve ser texto puro.
            """
        return "Prompt Padrão: Resuma os dados." # Fallback

    def generate_all_content(self, kpis, config, assets):
        """Gera o texto para todos os formatos de saída solicitados na configuração."""
        print("INFO: [Copywriter] Gerando conteúdo textual via API LLM...")
        content = {}
        requested_outputs = config.get('outputs', [])
        llm_provider = config.get('llm_provider', 'gemini')

        for output_type in requested_outputs:
            # Atualiza status no workflow para "Em desenvolvimento"
            content_id = f"{config['case_name'].replace(' ', '_')}_{output_type}"
            workflow_manager.update_job_status(content_id, "Em desenvolvimento")

            prompt = self._create_prompt(output_type, kpis, config)
            generated_text = self.llm_service.generate(prompt, provider=llm_provider)
            content[output_type] = generated_text

        print("INFO: [Copywriter] Conteúdo textual gerado.")
        return content


class ReportAssembler:
    """
    Agente 5: Montador de Relatórios
    Coleta todos os outputs e os monta no formato final para aprovação.
    """
    def assemble_for_review(self, content, assets, config):
        print("\n" + "="*80)
        print("PARA APROVAÇÃO HUMANA: CONTEÚDO GERADO PELO FRAMEWORK AGNÓSTICO")
        print("="*80 + "\n")

        for format_name, text in content.items():
            display(HTML(f"<h3>FORMATO: {format_name.replace('_',' ').upper()}</h3>"))

            if "report" in format_name:
                display(HTML(f'<img src="{assets["cover_image_url"]}" style="width: 300px; border: 1px solid #ccc;" />'))
                display(HTML(text))
            elif "blog" in format_name:
                display(HTML(f'<img src="{assets["blog_header_url"]}" style="width: 500px; border: 1px solid #ccc;" />'))
                display(HTML(text))
            elif "linkedin" in format_name:
                display(HTML(f'<img src="{assets["linkedin_infographic_url"]}" style="width: 300px; border: 1px solid #ccc;" />'))
                display(HTML(f"<pre style='white-space: pre-wrap; word-wrap: break-word; background-color: #f0f0f0; padding: 10px; border-radius: 5px;'>{text}</pre>"))
            elif "instagram" in format_name:
                display(HTML(f'<img src="{assets["instagram_map_url"]}" style="width: 300px; border: 1px solid #ccc;" />'))
                display(HTML(f"<pre style='white-space: pre-wrap; word-wrap: break-word; background-color: #f0f0f0; padding: 10px; border-radius: 5px;'>{text}</pre>"))

        print("\n" + "="*80)
        print("APROVAÇÃO SOLICITADA: Por favor, reveja o conteúdo gerado acima.")
        print("="*80 + "\n")


# --- Célula 4: Configuração do Caso de Uso e Orquestração ---

# Este dicionário de configuração é o ÚNICO item que precisa ser alterado
# para executar o framework para um novo caso de uso.
config_risco_desastres_rs = {
  "case_name": "Risco de Desastres Geohidrológicos",
  "ecosystem": "Climático",
  "region": "RS",
  # AÇÃO NECESSÁRIA: Cole o ID da sua planilha Google aqui
  "spreadsheet_id": "1hriMWNFvVtKn2i7OUaaQAV__ZNRCjqkcAea_2FzLwWY",
  "data_file_path": "/content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/03_gold/climate_risk_final_analysis.geoparquet",
  "llm_provider": "gemini", # Escolha entre 'gemini', 'openai', 'anthropic'
  "key_metrics": {
    "risk_category_col": "Risco_Ampliado_MCDA_Cat",
    "risk_score_col": "Risco_Ampliado_MCDA_Score",
    "trend_col": "Tendencia_Eventos_Climaticos_Extremos",
    "main_threat_col": "principal_ameaca",
    "lisa_cluster_col": "LAST10_YEARS_COUNT_lisa_labels"
  },
  # Lista de outputs a serem gerados
  "outputs": ["full_report", "report_summary", "blog_post", "linkedin_post", "instagram_post"]
}

# Variável global para o workflow manager, permitindo que outros agentes o acessem
workflow_manager = None

def run_framework(config):
    """Função principal que orquestra a execução dos agentes."""
    global workflow_manager

    print(f"--- INICIANDO FRAMEWORK PARA O CASO: {config['case_name']} EM {config['region']} ---")

    # 1. Gestor de Workflow
    if not config.get("spreadsheet_id") or "SUA_PLANILHA_AQUI" in config.get("spreadsheet_id"):
        print("ERRO: ID da Planilha Google não configurado. Por favor, atualize o 'spreadsheet_id' na configuração.")
        return
    workflow_manager = WorkflowManager(config['spreadsheet_id'])
    sheet_title = f"{config['region']}_{config['ecosystem']}_{config['case_name']}".replace(' ', '_')
    if not workflow_manager.setup_control_sheet(sheet_title):
        return
    workflow_manager.add_content_jobs(config['outputs'], config['case_name'])

    # 2. Sintetizador de Dados
    data_synthesizer = DataSynthesizer(config)
    if not data_synthesizer.load_data(): return
    kpis = data_synthesizer.extract_kpis()
    if not kpis: return

    # 3. Diretor de Arte
    art_director = MultimediaArtDirector()
    prompts = art_director.generate_visual_prompts(kpis, config)
    assets = art_director.generate_visual_assets(prompts)

    # 4. Gerador de Conteúdo
    copywriter = LLMCopywriter()
    content = copywriter.generate_all_content(kpis, config, assets)

    # 5. Monta para Revisão e Atualiza Workflow
    assembler = ReportAssembler()
    assembler.assemble_for_review(content, assets, config)
    for fmt in config['outputs']:
        content_id = f"{config['case_name'].replace(' ', '_')}_{fmt}"
        # Em um sistema real, o link poderia ser para um rascunho salvo (ex: Google Docs)
        workflow_manager.update_job_status(content_id, "Aguardando Aprovação", "Revisar output no notebook de execução.")


# --- Célula 5: Exemplo de Configuração para um Novo Caso de Uso ---

# Para demonstrar a natureza agnóstica do framework, veja como seria fácil
# configurá-lo para um caso de uso de "Desmatamento".
config_desmatamento_am = {
  "case_name": "Risco de Desmatamento Ilegal",
  "ecosystem": "Florestal",
  "region": "AM",
  # AÇÃO NECESSÁRIA: Cole o ID da sua planilha Google aqui
  "spreadsheet_id": "ID_DA_SUA_PLANILHA_AQUI",
  "data_file_path": "./desmatamento_final_analysis_am.geoparquet", # Caminho para os dados de desmatamento
  "llm_provider": "gemini",
  "key_metrics": {
    "risk_category_col": "Categoria_Risco_Desmatamento",
    "risk_score_col": "Score_Risco_Desmatamento",
    "trend_col": "Tendencia_Desmatamento",
    "main_threat_col": "Principal_Vetor_Desmatamento", # Ex: 'Grilagem', 'Garimpo'
    "lisa_cluster_col": "LISA_Cluster_Area_Desmatada_5a"
  },
   "outputs": ["full_report", "report_summary"]
}


# --- Célula 6: Execução ---

print("Framework de Agentes v3.0 definido.")
# Para executar, preencha o 'spreadsheet_id' na configuração
# e descomente a linha abaixo.
# Certifique-se de que o arquivo de dados existe no caminho especificado.
#
run_framework(config_risco_desastres_rs)

# Para1 executar o segundo caso de uso, bastaria chamar:
# run_framework(config_desmatamento_am)

Mounted at /content/drive/
Chaves de API carregadas com sucesso.
Framework de Agentes v3.0 definido.
--- INICIANDO FRAMEWORK PARA O CASO: Risco de Desastres Geohidrológicos EM RS ---


INFO: [WorkflowManager] Verificando/Configurando a aba de controlo: 'RS_Climático_Risco_de_Desastres_Geohidrológicos'


INFO: [WorkflowManager] Aba 'RS_Climático_Risco_de_Desastres_Geohidrológicos' já existe.
INFO: [WorkflowManager] Adicionando tarefas de conteúdo à Tabela de Controlo...
INFO: [WorkflowManager] 5 tarefas adicionadas com sucesso.
INFO: [DataSynthesizer] Carregando dados de: /content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/03_gold/climate_risk_final_analysis.geoparquet
INFO: [DataSynthesizer] Dados carregados e filtrados com sucesso.
INFO: [DataSynthesizer] Extraindo KPIs...
INFO: [DataSynthesizer] KPIs extraídos: {'total_municipalities': 499, 'percent_high_risk': 40, 'percent_growing_trend': 62, 'dominant_threat': 'Chuvas Intensas', 'top_risk_municipalities': ['PONTE PRETA', 'GENTIL', 'BARRA DO RIO AZUL']}
INFO: [ArtDirector] Gerando prompts para visuais...
INFO: [ArtDirector] Prompts gerados.
INFO: [ArtDirector] SIMULANDO geração de imagens a partir dos prompts...
INFO: [ArtDirector] URLs de ativos visuais simulados retornados.
INFO: [Copywriter] Gerando conteúdo t


APROVAÇÃO SOLICITADA: Por favor, reveja o conteúdo gerado acima.

INFO: [WorkflowManager] Atualizando estado de 'Risco_de_Desastres_Geohidrológicos_full_report' para 'Aguardando Aprovação'...
INFO: [WorkflowManager] Estado de 'Risco_de_Desastres_Geohidrológicos_full_report' atualizado com sucesso.
INFO: [WorkflowManager] Atualizando estado de 'Risco_de_Desastres_Geohidrológicos_report_summary' para 'Aguardando Aprovação'...
INFO: [WorkflowManager] Estado de 'Risco_de_Desastres_Geohidrológicos_report_summary' atualizado com sucesso.
INFO: [WorkflowManager] Atualizando estado de 'Risco_de_Desastres_Geohidrológicos_blog_post' para 'Aguardando Aprovação'...
INFO: [WorkflowManager] Estado de 'Risco_de_Desastres_Geohidrológicos_blog_post' atualizado com sucesso.
INFO: [WorkflowManager] Atualizando estado de 'Risco_de_Desastres_Geohidrológicos_linkedin_post' para 'Aguardando Aprovação'...
INFO: [WorkflowManager] Estado de 'Risco_de_Desastres_Geohidrológicos_linkedin_post' atualizado com suce

In [ ]:
from google.colab import userdata
import google.generativeai as genai

try:
    api_key = userdata.get('GOOGLE_API_KEY')
    if api_key:
        genai.configure(api_key=api_key)
        model = genai.GenerativeModel('gemini-2.5-flash')
        response = model.generate_content('Teste de conexão.')
        print('✅ Sucesso: API Key configurada e validada!')
        print(f'Resposta do teste: {response.text}')
    else:
        print('❌ Erro: A variável GOOGLE_API_KEY não foi encontrada nos Secrets do Colab.')
except Exception as e:
    print(f'❌ Erro ao validar API Key: {e}')
    print('\nCertifique-se de que o segredo GOOGLE_API_KEY existe e o acesso ao notebook está ativado.')

✅ Sucesso: API Key configurada e validada!
Resposta do teste: Olá! Teste de conexão recebido com sucesso.

Estou online e pronto para ajudar. Em que posso te auxiliar hoje?
